# P1-SHAPE frozen full backbone screen

Run only after both P1 smoke receipts pass. Select a T4 GPU, run once per backbone, and keep the same Drive output directory. Units resume by backbone, mechanism, and seed. This notebook cannot compute the aggregate paper gate.

In [ ]:
BACKBONE = 'chronos_2'  # then rerun the notebook with 'timesfm_3'
assert BACKBONE in {'chronos_2', 'timesfm_3'}
print('Selected frozen backbone:', BACKBONE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os, json, torch

REPO = Path('/content/tsfm-covariate-faithfulness')
REPO_URL = 'https://github.com/FlyMe2star/tsfm-covariate-faithfulness.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements/colab-base.txt')], check=True)
model_requirements = REPO / ('requirements/chronos2.txt' if BACKBONE == 'chronos_2' else 'requirements/timesfm3.txt')
if BACKBONE == 'timesfm_3':
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'timesfm'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(model_requirements)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
assert torch.cuda.is_available(), 'Select a T4 GPU and restart the runtime.'
print('Repository commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from covfaith.config import load_yaml, verify_config_lock
from covfaith.adapters import Chronos2Adapter, TimesFM3Adapter

config_path = REPO / 'configs/p1_shape/covintervene_shape_p1.yaml'
lock_path = REPO / 'configs/p1_shape/covintervene_shape_p1.lock.json'
config_hash = verify_config_lock(config_path, lock_path)
config = load_yaml(config_path)
model = next(item for item in config['models'] if item['id'] == BACKBONE)
if BACKBONE == 'chronos_2':
    adapter = Chronos2Adapter.from_pretrained(model['checkpoint'], model['revision'], device='cuda', batch_size=128)
else:
    adapter = TimesFM3Adapter.from_pretrained(model['checkpoint'], model['revision'], device='cuda', per_core_batch_size=16)
print('Frozen P1-SHAPE config hash:', config_hash[:12])

In [ ]:
from covfaith.p1_shape import run_shape_backbone_units

OUTPUT_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p1_shape_v1')
report = run_shape_backbone_units(REPO, adapter, OUTPUT_ROOT)
assert report['scientific_gate_computed'] is False
assert report['completed_unit_count'] == 12
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Full units:', OUTPUT_ROOT)